In [0]:
-- =========================================================
-- Silver: Patients
-- deduplication, drop personally identifiable information,
-- derive age, age band, decreased flag
-- =========================================================

CREATE OR REPLACE TABLE health_lakehouse.silver.patients AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) AS rn FROM health_lakehouse.bronze.patients
),
cleaned AS (
    SELECT
    Id AS patient_id,
    BIRTHDATE AS birth_date,
    DEATHDATE AS death_date,
    MARITAL AS marital_status,
    RACE AS race,
    ETHNICITY AS ethnicity,
    GENDER AS gender,
    CITY AS city,
    STATE AS state,
    COUNTY AS county,
    ZIP AS zip,
    LAT AS latitude,
    LON AS longitude,
    HEALTHCARE_EXPENSES AS healthcare_expenses,
    HEALTHCARE_COVERAGE AS healthcare_coverage,
    INCOME AS income,
    -- age at death if deceased, else age today
    CAST(FLOOR(DATEDIFF(COALESCE(DEATHDATE, current_date()),BIRTHDATE)/365.25)AS INT) AS age,
    (DEATHDATE IS NOT NULL) AS is_deceased
    FROM deduplicated
    WHERE rn=1 AND Id IS NOT NULL
),
banded AS (
    SELECT *,
        CASE
            WHEN age < 18 THEN '0-17'
            WHEN age < 35 THEN '18-34'
            WHEN age < 50 THEN '35-49'
            WHEN age < 65 THEN '50-64'
            WHEN age < 80 THEN '65-79'
            ELSE '80+'
        END AS age_band
    FROM cleaned
)
SELECT * FROM banded;


In [0]:
-- =========================================================
-- Silver: encounters
-- deduplication, rename, derive lenght of stay,
-- validate timestamps
-- =========================================================

CREATE OR REPLACE TABLE health_lakehouse.silver.encounters AS
WITH deduplicated AS(
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) AS rn
    FROM health_lakehouse.bronze.encounters
),
cleaned AS (
    SELECT
    Id AS encounter_id,
    PATIENT AS patient_id,
    ORGANIZATION AS organization_id,
    PROVIDER AS provider_id,
    PAYER AS payer_id,
    START AS start_ts,
    STOP AS stop_ts,
    ENCOUNTERCLASS AS encounter_class,
    CAST (CODE AS STRING) AS encounter_code,
    DESCRIPTION AS encounter_desc,
    BASE_ENCOUNTER_COST AS base_cost,
    TOTAL_CLAIM_COST as total_calim_cost,
    PAYER_COVERAGE AS payer_coverage,
    CAST(REASONCODE AS STRING) AS reason_code,
    REASONDESCRIPTION AS reason_desc,
    -- lenght of stay in hours (encounters can be less than a day)
    ROUND((UNIX_TIMESTAMP(STOP)-UNIX_TIMESTAMP(START))/3600.0,2) AS los_hours
    FROM deduplicated
    WHERE rn =1
        AND Id IS NOT NULL
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
)
SELECT * FROM cleaned
WHERE stop_ts IS NOT NULL OR stop_ts >= start_ts; -- to drop impossible time-travel rows


In [0]:
-- checking new tables.
SELECT COUNT(*) AS patient_rows FROM health_lakehouse.silver.patients;
SELECT * FROM health_lakehouse.silver.patients LIMIT 5;

SELECT COUNT(*) AS encounter_rows FROM health_lakehouse.silver.encounters;
SELECT * FROM health_lakehouse.silver.encounters LIMIT 5;

In [0]:
-- =========================================================
-- Silver: conditions
-- No natural primary key (create composite key on depuplicated)
-- Derives:is_active, duration days
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.conditions AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    )AS rn
    FROM health_lakehouse.bronze.conditions
),
cleanded AS (
SELECT
    PATIENT AS patient_id,
    ENCOUNTER AS encounter_id,
    CAST(CODE AS STRING) AS condtion_code,
    DESCRIPTION AS condition_desc,
    SYSTEM AS code_system,
    START AS onset_date,
    STOP AS resolved_date,
    (STOP IS NULL ) as is_active,
    DATEDIFF (COALESCE(STOP, current_date()),START) AS duration_days
FROM deduplicated
WHERE rn =1
    AND PATIENT IS NOT NULL
    AND START IS NOT NULL
)
SELECT * FROM cleanded
WHERE resolved_date is NULL OR resolved_date >= onset_date;

In [0]:
-- =========================================================
-- Silver: medications
-- Derives:duration_days (null = ongoing prescription)
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.medications AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.medications
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        PAYER AS payer_id,
        CAST(CODE AS STRING) AS medication_code,
        DESCRIPTION AS medication_desc,
        START AS start_ts,
        STOP AS stop_ts,
        BASE_COST AS base_cost,
        PAYER_COVERAGE AS payer_coverage,
        DISPENSES AS dispenses,
        TOTALCOST AS total_cost,
        CAST(REASONCODE AS STRING) AS reason_code,
        REASONDESCRIPTION AS reason_desc,
        DATEDIFF (STOP,START) AS duration_days,
        (STOP IS NULL) AS is_ongoing
    FROM deduplicated
    WHERE rn =1
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
    
)
SELECT * FROM cleaned
WHERE stop_ts IS NULL OR stop_ts >= start_ts;

In [0]:
-- =========================================================
-- Silver: procedures
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.procedures AS
WITH deduplicated AS(
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.procedures
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        CAST(CODE AS STRING) AS procedure_code,
        DESCRIPTION AS procedure_desc,
        SYSTEM AS code_system,
        START AS start_ts,
        STOP AS stop_ts,
        BASE_COST AS base_cost,
        CAST(REASONCODE AS STRING) AS reason_code,
        REASONDESCRIPTION AS reason_desc
    FROM deduplicated
    WHERE rn =1
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
)
SELECT * FROM cleaned;

In [0]:
-- =========================================================
-- Silver: Observations
-- Tall key-value table. VALUE holds both numeric and text result,
-- TRY_CAST into value_numeric and retain the raw string
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.observations AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER() OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, DATE
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.observations
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        DATE AS observed_ts,
        CATEGORY AS category,
        CODE AS observation_code,
        DESCRIPTION AS observation_desc,
        VALUE AS value_raw,
        TRY_CAST(VALUE AS DOUBLE) AS value_numeric,
        UNITS AS units,
        TYPE as value_type
    FROM deduplicated
    WHERE rn=1
        AND PATIENT IS NOT NULL
        AND DATE IS NOT NULL
)
SELECT * FROM cleaned;

In [0]:
-- ============================================================
-- SILVER: observations_vitals_labs (curated narrow table)
-- Vitals + basic metabolic panel + renal panel (creatinine, BUN,
-- eGFR) + HbA1c + dialysis marker
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.observations_vitals_labs AS
WITH curated_codes AS (
  SELECT code, grp FROM VALUES
    -- vitals
    ('8480-6',  'vital: systolic_bp'),
    ('8462-4',  'vital: diastolic_bp'),
    ('29463-7', 'vital: body_weight'),
    ('8302-2',  'vital: body_height'),
    ('39156-5', 'vital: bmi'),
    ('8867-4',  'vital: heart_rate'),
    ('9279-1',  'vital: respiratory_rate'),
    ('72514-3', 'vital: pain_score'),
    -- basic metabolic panel
    ('38483-4', 'lab: creatinine'),
    ('2339-0',  'lab: glucose'),
    ('2947-0',  'lab: sodium'),
    ('6298-4',  'lab: potassium'),
    ('2069-3',  'lab: chloride'),
    ('49765-1', 'lab: calcium'),
    ('20565-8', 'lab: co2_total'),
    -- renal panel (BUN under two harmonized codes; eGFR)
    ('6299-2',  'lab: bun'),
    ('3094-0',  'lab: bun'),
    ('33914-3', 'lab: egfr'),
    -- glycemic control
    ('4548-4',  'lab: hba1c'),
    -- dialysis marker
    ('74006-8', 'renal: dialysis_weight_diff')
  AS t(code, grp)
)
SELECT
  o.patient_id,
  o.encounter_id,
  o.observed_ts,
  o.observation_code,
  o.observation_desc,
  c.grp                AS measure_group,
  o.value_numeric,
  o.units
FROM health_lakehouse.silver.observations o
INNER JOIN curated_codes c
  ON o.observation_code = c.code
WHERE o.value_numeric IS NOT NULL;

In [0]:
-- investigating vitals to see if they fall in appropriate standard ranges.
SELECT measure_group, COUNT(*) AS n,
       ROUND(AVG(value_numeric),1) AS avg_val,
       MIN(value_numeric) AS min_val, MAX(value_numeric) AS max_val
FROM health_lakehouse.silver.observations_vitals_labs
GROUP BY measure_group ORDER BY n DESC;

In [0]:
-- creatinine max of 115.4mg/dl flag!
-- creatinine usually tops out at 15-20 mg/dl even in end stage
-- possible unit swap from mg/dl to umol/l or wrong value inputs

SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN value_numeric > 20 THEN 1 ELSE 0 END) AS implausible,
  ROUND(100.0 * SUM(CASE WHEN value_numeric > 20 THEN 1 ELSE 0 END) / COUNT(*), 3) AS pct_implausible,
  PERCENTILE(value_numeric, 0.50) AS p50,
  PERCENTILE(value_numeric, 0.95) AS p95,
  PERCENTILE(value_numeric, 0.99) AS p99
FROM health_lakehouse.silver.observations_vitals_labs
WHERE measure_group = 'lab: creatinine';

In [0]:
-- Do the outliers share a unit or description?
SELECT units, observation_desc, COUNT(*) AS n, MAX(value_numeric) AS max_val
FROM health_lakehouse.silver.observations_vitals_labs
WHERE measure_group = 'lab: creatinine' AND value_numeric > 20
GROUP BY units, observation_desc;

In [0]:
-- ============================================================
-- Silver: observations_vitals_labs (curated narrow table)
-- Vitals + metabolic + renal panel + HbA1c + dialysis marker.
-- Harmonizes duplicate BUN codes; applies clinical plausibility
-- bounds (implausible values nulled, retained + flagged).
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.observations_vitals_labs AS
WITH curated_codes AS (
  SELECT code, grp FROM VALUES
    ('8480-6','vital: systolic_bp'),
    ('8462-4','vital: diastolic_bp'),
    ('29463-7','vital: body_weight'),
    ('8302-2','vital: body_height'),
    ('39156-5','vital: bmi'),
    ('8867-4','vital: heart_rate'),
    ('9279-1','vital: respiratory_rate'),
    ('72514-3','vital: pain_score'),
    ('38483-4','lab: creatinine'),
    ('2339-0','lab: glucose'),
    ('2947-0','lab: sodium'),
    ('6298-4','lab: potassium'),
    ('2069-3','lab: chloride'),
    ('49765-1','lab: calcium'),
    ('20565-8','lab: co2_total'),
    ('6299-2', 'lab: bun'),
    ('3094-0', 'lab: bun'),
    ('33914-3','lab: egfr'),
    ('4548-4','lab: hba1c'),
    ('74006-8','renal: dialysis_weight_diff')
  AS t(code, grp)
),
-- Clinical plausibility bounds, applied per measure.
-- Rationale documented in README; values outside bounds are
-- nulled (not dropped) so the record of the test is preserved.
bounds AS (
  SELECT grp, lo, hi FROM VALUES
    ('lab: creatinine',0.1,20.0),   -- >20 mg/dL not survivable
    ('lab: egfr',1.0, 200.0),
    ('lab: hba1c',2.0,20.0),
    ('lab: bun',1.0,200.0),
    ('vital: systolic_bp',50.0,300.0),
    ('vital: diastolic_bp',20.0,200.0),
    ('vital: bmi',10.0,80.0),
    ('vital: heart_rate',20.0,250.0)
  AS t(grp, lo, hi)
),
joined AS (
  SELECT
    o.patient_id,
    o.encounter_id,
    o.observed_ts,
    o.observation_code,
    o.observation_desc,
    c.grp AS measure_group,
    o.value_numeric AS value_raw_numeric,
    o.units
  FROM health_lakehouse.silver.observations o
  INNER JOIN curated_codes c
    ON o.observation_code = c.code
  WHERE o.value_numeric IS NOT NULL
),
validated AS (
  SELECT
    j.*,
    -- LEFT JOIN: measures without bounds pass through unchanged
    CASE
      WHEN b.grp IS NULL THEN TRUE
      WHEN j.value_raw_numeric BETWEEN b.lo AND b.hi THEN TRUE
      ELSE FALSE
    END AS is_plausible
  FROM joined j
  LEFT JOIN bounds b
    ON j.measure_group = b.grp
)
SELECT
  patient_id,
  encounter_id,
  observed_ts,
  observation_code,
  observation_desc,
  measure_group,
  CASE WHEN is_plausible THEN value_raw_numeric END AS value_numeric,
  value_raw_numeric,
  is_plausible,
  units
FROM validated;

In [0]:
SELECT measure_group,
       COUNT(*) AS n,
       SUM(CASE WHEN NOT is_plausible THEN 1 ELSE 0 END) AS excluded,
       ROUND(AVG(value_numeric), 2) AS avg_clean,
       MAX(value_numeric) AS max_clean
FROM health_lakehouse.silver.observations_vitals_labs
GROUP BY measure_group
ORDER BY excluded DESC;